# Clase 3 — Práctica Guiada
## ¿Cómo viene el negocio?
### Maestría en Fintech · ITBA · 2026

---

**Esta notebook la resolvemos juntos en clase.** Después vas a tener una segunda notebook
—la de Práctica Individual— con preguntas nuevas sobre el mismo dataset, para resolver solo.

**Las preguntas de hoy:**
> *¿En qué gastan nuestros clientes? ¿Y cómo evolucionó ese gasto a lo largo de 2024?*

La semana pasada teníamos **1.000 clientes**, una fila por cliente. Hoy tenemos **~29.000 transacciones**,
una fila por movimiento. Esa tabla no se mira: se resume.

---
## ⚠️ Antes de escribir una sola línea: hacé tu propia copia

Esta notebook es **el original del curso**. Si escribís acá, se pisan entre todos y se pierde el trabajo.

1. Menú **`Archivo`** → **`Guardar una copia en Drive`**
2. Se abre una pestaña nueva llamada *Copia de Clase_3_Practica_Guiada.ipynb* — **esa es tuya**
3. Hacé clic en el nombre, arriba a la izquierda, y renombrala: **`Clase3_Guiada_TuNombre`**

Cerrá la pestaña del original y trabajá siempre en la tuya.

---
## Setup

Ejecutá la celda con **`Shift + Enter`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

print('Librerías cargadas correctamente')

---
# Paso 1 · Cargar y reconocer la tabla

Mismo movimiento que la clase pasada: leer el CSV y mirar qué hay adentro.

In [ ]:
# El dataset se descarga solo desde el repo del curso.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

df = pd.read_csv(DATOS + 'transacciones.csv')

print(f'{len(df):,} transacciones')
df.head()

In [ ]:
df.info()

**Cómo se lee cada fila:** una transacción. Quién (`cliente_id`, `tarjeta_id`), cuándo (`fecha`),
cuánto (`monto_ars`), en qué (`categoria`), con qué (`tipo`: débito o crédito) y si pasó (`aprobada`: 1 sí, 0 rechazada).

**Mirá el tipo de `fecha`:** dice `object` (en algunas versiones, `str`). Es **texto**, no una fecha. Por ahora no importa —
en el Paso 4 lo vamos a necesitar y lo vamos a arreglar.

---
# Paso 2 · Agrupar: ¿en qué gastan?

La lógica de `groupby` es siempre la misma, en este orden:

**agrupar por algo → elegir qué medir → cómo resumirlo**

```python
df.groupby('categoria')['monto_ars'].sum()
#          ↑ agrupar      ↑ qué medir   ↑ cómo
```

> Es una tabla dinámica de Excel escrita en una línea: `categoria` va a *Filas*, `monto_ars` a *Valores*, y `.sum()` es *Suma de*.

In [ ]:
# Gasto total por categoría, de mayor a menor (en millones)
gasto = df.groupby('categoria')['monto_ars'].sum().sort_values(ascending=False)
(gasto / 1e6).round(1)

**Pregunta para el grupo antes de seguir:** ¿cuál es la categoría "más importante" del negocio?

Ahora contemos cuántas veces se usa cada una. Misma línea, **otra función de agregación**.

In [ ]:
# Cantidad de transacciones por categoría
df.groupby('categoria')['monto_ars'].count().sort_values(ascending=False)

> **El ranking cambió.** Por plata gana `Viajes`; por cantidad gana `Supermercado`.
> Ninguno de los dos está mal: responden preguntas distintas. **La función de agregación que elegís
> es una decisión de negocio**, no un detalle técnico.

### Varias métricas de una vez: `.agg()`

Para no elegir a ciegas, las ponemos juntas en una sola tabla.

In [ ]:
resumen = df.groupby('categoria').agg(
    transacciones=('transaccion_id', 'count'),
    monto_total=('monto_ars', 'sum'),
    ticket_promedio=('monto_ars', 'mean'),
    ticket_mediano=('monto_ars', 'median'),
).sort_values('monto_total', ascending=False)

resumen.round(0)

**Cómo se lee:** cada fila es una categoría, cada columna una pregunta distinta sobre ella.

- `Viajes`: **pocas transacciones, muy grandes** — el ticket es 4 o 5 veces el de Supermercado
- `Supermercado`: **muchas transacciones, chicas** — es el hábito cotidiano
- En todas, el ticket promedio es mayor que el mediano. ¿Les suena? Es la misma asimetría del balance de la Clase 2

> Un producto que vive de la frecuencia y uno que vive del ticket se gestionan distinto.
> Solo mirando las dos columnas juntas se distingue cuál es cuál.

In [ ]:
# Lo mismo, en un gráfico
(gasto / 1e6).sort_values().plot(kind='barh', color='steelblue')
plt.title('Gasto total por categoría — 2024')
plt.xlabel('Millones ARS')
plt.ylabel('')
plt.show()

---
# Paso 3 · Pivot table: dos dimensiones a la vez

`groupby` cruza por una dimensión. Cuando querés **filas y columnas**, es una `pivot_table` —
los mismos cuatro campos que la tabla dinámica de Excel.

| Excel | pandas |
|---|---|
| Filas | `index` |
| Columnas | `columns` |
| Valores | `values` |
| Resumir por | `aggfunc` |

In [ ]:
# Gasto por categoría y tipo de tarjeta (millones)
pivot = df.pivot_table(
    values='monto_ars',
    index='categoria',
    columns='tipo',
    aggfunc='sum'
) / 1e6

pivot.round(1)

Los montos absolutos cuestan compararlos: las categorías tienen tamaños muy distintos.
La pregunta útil es otra: **dentro de cada categoría, ¿qué parte se paga con crédito?**

In [ ]:
# Participación de cada tipo dentro de cada categoría (cada fila suma 100%)
mix = pivot.div(pivot.sum(axis=1), axis=0) * 100
mix = mix.sort_values('crédito', ascending=False)

sns.heatmap(mix, annot=True, fmt='.0f', cmap='Blues', cbar_kws={'label': '% del gasto de la categoría'})
plt.title('¿Con qué se paga cada categoría? (%)')
plt.ylabel('')
plt.xlabel('')
plt.show()

**Lectura de negocio:**

- `Viajes`, `Indumentaria`, `E-commerce` → se pagan **con crédito**: la gente financia lo grande y lo que compra online
- `Transferencia`, `Servicios`, `Combustible`, `Supermercado` → **débito**: el gasto recurrente se paga con lo que hay

> Si el negocio vive del financiamiento, las promociones en cuotas van a `Viajes` e `Indumentaria`,
> no a `Supermercado`. Esa decisión sale de esta tabla.

---
# Paso 4 · Fechas: de texto a tiempo

Hasta acá la fecha no la usamos. Para hablar de evolución hay que convertirla.

In [ ]:
# Antes: texto
print(type(df['fecha'].iloc[0]))

df['fecha'] = pd.to_datetime(df['fecha'])

# Después: un momento en el tiempo
print(type(df['fecha'].iloc[0]))
print(f"Desde {df['fecha'].min():%d/%m/%Y} hasta {df['fecha'].max():%d/%m/%Y}")

Una vez convertida, la fecha "sabe" cosas de sí misma. Se le piden con `.dt`:

In [ ]:
df['mes'] = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter
df['dia_semana'] = df['fecha'].dt.day_name()

df[['fecha', 'mes', 'trimestre', 'dia_semana']].head()

> **La regla:** una fecha que sigue siendo texto no se puede ordenar bien, ni restar, ni agrupar por mes.
> Convertila siempre, apenas cargás el dataset.

---
# Paso 5 · Resample: la serie mensual

`resample` es un `groupby` pensado para el tiempo: agrupa por **período**. Cambiando una letra cambiás el zoom.

| Código | Frecuencia |
|---|---|
| `'D'` | diaria |
| `'W'` | semanal |
| `'ME'` | mensual |
| `'QE'` | trimestral |

Necesita que la fecha sea el índice de la tabla: por eso el `set_index('fecha')`.

In [ ]:
mensual = df.set_index('fecha').resample('ME').agg(
    monto=('monto_ars', 'sum'),
    transacciones=('transaccion_id', 'count'),
)
mensual['monto_M'] = (mensual['monto'] / 1e6).round(1)

mensual

In [ ]:
mensual['monto_M'].plot(marker='o', color='steelblue', linewidth=2)
plt.title('Gasto mensual — 2024')
plt.ylabel('Millones ARS')
plt.xlabel('')
plt.show()

**Las tres preguntas ante cualquier gráfico:** ¿qué forma tiene? ¿es lo que esperaba? ¿qué decisión cambia?

Miremos **agosto y septiembre**: el gasto cae dos meses seguidos, casi 10% y 12%.

> Pregunta para el grupo: **si sos el gerente comercial y ves esto en septiembre, ¿escribís una alarma?**

---
# Paso 6 · Promedio móvil: la señal detrás del ruido

Un mes suelto sube y baja por mil razones: vacaciones, un feriado, un cliente grande.
El **promedio móvil** reemplaza cada mes por el promedio de los últimos N. La línea se suaviza y aparece la tendencia.

In [ ]:
# Promedio de los últimos 3 meses
mensual['media_movil_3m'] = mensual['monto_M'].rolling(3).mean().round(1)

mensual[['monto_M', 'media_movil_3m']]

**Fijate las dos primeras filas: `NaN`.** No es un error. Con una ventana de 3, el primer promedio
recién se puede calcular en marzo: antes no hay tres meses para promediar. Es aritmética.

In [ ]:
plt.plot(mensual.index, mensual['monto_M'], marker='o', alpha=0.4, color='steelblue', label='Gasto mensual')
plt.plot(mensual.index, mensual['media_movil_3m'], linewidth=3, color='darkblue', label='Media móvil 3 meses')
plt.title('Gasto mensual y tendencia — 2024')
plt.ylabel('Millones ARS')
plt.legend()
plt.show()

**Volvamos a septiembre.** La serie cruda cae fuerte. La media móvil apenas se aplana y sigue arriba de junio.

> Julio fue un pico (vacaciones de invierno), y agosto y septiembre son la vuelta a lo normal.
> **La caída es contra un mes excepcional, no contra la tendencia.**
> Al comité se le reporta la tendencia. El zigzag mes a mes no es información: es ansiedad.

---
# Paso 7 · La trampa de los pesos

El gasto se **duplicó** entre enero y diciembre. ¿El negocio se duplicó?

Para comparar dos series de escalas distintas, las llevamos a **base 100**: enero = 100.

In [ ]:
base100 = pd.DataFrame({
    'Monto (ARS)': mensual['monto'] / mensual['monto'].iloc[0] * 100,
    'Cantidad de transacciones': mensual['transacciones'] / mensual['transacciones'].iloc[0] * 100,
})

base100.plot(marker='o', linewidth=2)
plt.axhline(100, color='gray', linestyle='--')
plt.title('Monto vs cantidad — base enero = 100')
plt.ylabel('Índice')
plt.show()

base100.round(0).tail(3)

**Lectura:** en diciembre el monto está en ~208 y la cantidad en ~140. **Casi toda la brecha es precio**, no actividad:
con inflación, cada compra cuesta más pesos aunque la gente compre lo mismo.

> En Argentina, **toda serie en pesos crece**. Antes de celebrar un crecimiento, mirá la cantidad.
> Es la trampa más común de cualquier análisis financiero hecho acá.

---
# Paso 8 · El caso: ¿qué le decimos al gerente comercial?

Juntamos lo que aprendimos: **agrupar + tiempo**. ¿Cómo evolucionaron las categorías principales mes a mes?

In [ ]:
top4 = gasto.head(4).index

evol = df.pivot_table(
    values='monto_ars',
    index=pd.Grouper(key='fecha', freq='ME'),
    columns='categoria',
    aggfunc='sum'
)[top4] / 1e6

evol.plot(marker='o', linewidth=2)
plt.title('Top 4 categorías, mes a mes (millones ARS)')
plt.ylabel('Millones ARS')
plt.xlabel('')
plt.legend(title='Categoría', bbox_to_anchor=(1.01, 1))
plt.show()

**¿Qué vemos?**

- `Viajes` es la más volátil: un pico enorme en **julio** y otra subida en **noviembre–diciembre**
- `Supermercado` e `Indumentaria` crecen parejo, con el salto de **diciembre** (aguinaldo y fiestas)
- `Transferencia` no tiene estacionalidad clara

Una última mirada antes de recomendar: **¿quién está gastando?**

In [ ]:
clientes_mes = df.set_index('fecha').resample('ME')['cliente_id'].nunique()

print(f'Clientes que operaron en enero:     {clientes_mes.iloc[0]}')
print(f'Clientes que operaron en diciembre: {clientes_mes.iloc[-1]}')
print(f'Variación: {clientes_mes.iloc[-1] / clientes_mes.iloc[0] - 1:.1%}')

> **El monto se duplicó, pero cada mes operan menos clientes.** El negocio se está concentrando en menos gente.
> Es exactamente lo que la Clase 2 nos dejó planteado: los que se van, dejan de transaccionar antes de irse.

### Del análisis a la recomendación — tres bullets, no una tabla

1. **El volumen:** el gasto pasó de $X M en enero a $Y M en diciembre, pero la cantidad de transacciones creció mucho menos: gran parte es precio
2. **La estacionalidad:** julio y diciembre son los picos; la caída de agosto–septiembre es la vuelta a lo normal, no una alarma
3. **La señal:** los clientes activos bajaron Z% en el año — el crecimiento en pesos está tapando una base que se achica

**Escribamos los tres bullets juntos, en voz alta, con los números que acabamos de sacar.**

---
## Hasta acá la práctica guiada

**Lo que hiciste hoy:** agrupaste por una y por dos dimensiones, elegiste funciones de agregación,
convertiste fechas, armaste una serie mensual, la suavizaste con un promedio móvil y la corregiste por la trampa de los pesos.

**Lo que sigue:** abrí la notebook **`Clase_3_Practica_Individual.ipynb`**.
Mismo dataset, preguntas nuevas, y esta vez lo resolvés vos con Gemini de copiloto.

> Recordá: primero **`Archivo → Guardar una copia en Drive`**. Siempre.